# Do two groups of cells differ in shape?

You have two groups of cells, **A** and **B**, and you want to know whether
they differ in shape, and by how much. Each group can come with a **reference**
group: the cells around it or the tissue it sits in. Then the question becomes
the more careful one: does A differ from *its own* reference, and does B differ
from *its own* reference in the same way? The list of those differences, one per
measurement, is the group's **departure profile**. Choose each reference from the
same stacks as its group (for example the rest of the tissue in those stacks);
otherwise a difference between stacks looks like a difference between groups.

What you do:

1. Point the notebook at your embedding table (`shape_embedding.csv` or `niche_embedding.csv`).
2. Define the groups (the EDIT cell, section 1). There are three ways: by a **column and its values**, by a **list of cells in a CSV file**, or by a **rule** on a column (for example "sphericity above 0.8").
3. Run the cells from top to bottom. **The first time, leave `GROUP_SPEC` empty:** the notebook prints an overview of your clusters (section 3) and then stops at section 4 asking you to fill it in. That stop is expected. Fill in `GROUP_SPEC` and run again from the top.

What you get, in the output folder: `comparison_tables/` (every number as a CSV)
and `figures/` (SVG, editable, and PNG), and a plain-language summary at the end.

## How to read the statistics (read this first)

* Cells within a stack are **not independent**: neighbouring cells resemble each other. For that reason this notebook reports **no cell-level p-values**. It reports **effect sizes with bootstrap 95% confidence intervals**, which are descriptive.
* The bootstrap resamples **cells** within each group. The intervals therefore reflect the sampling of cells, **not** differences between replicate stacks or samples. They are narrower than an interval across replicates would be. With several stacks, the replicate plots (one dot per stack) show whether a result repeats.
* If your groups were **picked from clusters**, and those clusters were built from the same shape measurements that are compared here, some separation between groups is expected **by construction**. The informative part is the comparison *relative to the references*: A against B, next to reference against reference, and the two departure profiles against each other.
* **Cliff's delta** = P(a > b) - P(a < b), from -1 to 1. 0 means no shift. Rough guide: |delta| < 0.147 negligible, < 0.33 small, < 0.474 medium, otherwise large. Delta depends only on ranks, so a log transform does not change it.
* Features measured along the image axes (for example z extent or elongation along z) mix how elongated a cell is with how it is tilted. `major_axis_z_angle` is the tilt itself, and `inertia_elongation` or `axis_minor_length` describe the cell's own shape whatever its orientation.

In [ ]:
# Make the helper file next to this notebook importable (works from the repo root or from this folder).
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "tutorials" / "analysis", Path.cwd().parent):
    if (_p / "_helpers.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find _helpers.py. Open this notebook from the repository root or from tutorials/analysis.")

import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _helpers as h

warnings.filterwarnings("ignore", category=FutureWarning)
h.apply_style()
print("Ready.")

## 1. Edit this cell

- `EMBEDDING_CSV`: `shape_embedding.csv` (from the first notebook) or `niche_embedding.csv`. It needs the shape measurements, the principal components (`PC1`...; the principal components written by the first notebook) and, for the map, `umap_1` and `umap_2`.
- `METADATA_CSV`: optional table with one row per stack and any columns you like.
- `N_BOOT`, `MAX_CELLS_FOR_BOOTSTRAP`: the bootstrap is the slow part. Its time grows with the number of cells per group, so a group larger than `MAX_CELLS_FOR_BOOTSTRAP` is cut to a random subset for the bootstrap only (the delta itself uses every cell, and the intervals become a little wider). The notebook prints an estimate of the time before it starts.
- `OVERVIEW_COLUMN`: a column to print an overview of (section 3), to help you choose groups. A cluster column such as `leiden_r1.0` works well.

**Defining the groups.** There are four roles: `A`, `A_ref` (the reference of A), `B`, `B_ref` (the reference of B). Only `A` and `B` are required. Set a role to `None` to leave it out. `GROUP_MODE` decides what you type for each role in `GROUP_SPEC`:

| `GROUP_MODE` | What you type per role | Example |
|---|---|---|
| `"column"` | a list of values of `GROUP_COLUMN` | `{"A": [3, 7], "A_ref": [0], "B": [5], "B_ref": [1, 2]}` with `GROUP_COLUMN = "leiden_r1.0"` |
| `"csv"` | the name of a group in `GROUP_CSV` (columns `stack_name`, `label`, `group`) | `{"A": "region_one", "A_ref": "background_one", "B": "region_two", "B_ref": None}` |
| `"rule"` | a rule on the columns of the table | `{"A": "sphericity > 0.8", "B": "area > 500 and sphericity < 0.6", "A_ref": None, "B_ref": None}` |

In a rule, put a column name in backticks if it contains `-` or a space, for example ``"`centroid-0` < 30 and sphericity > 0.7"``. The groups must not share cells. `GROUP_NAMES` are the names shown on the figures.

In [ ]:
# ---------------- EDIT ME ----------------
EMBEDDING_CSV = "shape_output/shape_embedding.csv"
METADATA_CSV = None            # for example "metadata.csv"
OUTPUT_DIR = "comparison_output"
OVERVIEW_COLUMN = "leiden_r1.0"    # printed in section 3 to help you pick groups

GROUP_MODE = "column"          # "column", "csv" or "rule"
GROUP_COLUMN = "leiden_r1.0"   # used when GROUP_MODE = "column"
GROUP_CSV = None               # used when GROUP_MODE = "csv": a CSV with stack_name, label, group
GROUP_SPEC = {
    "A": None,                 # fill in, see the table above
    "A_ref": None,             # optional reference of A
    "B": None,                 # fill in
    "B_ref": None,             # optional reference of B
}
GROUP_NAMES = {"A": "group A", "A_ref": "A reference", "B": "group B", "B_ref": "B reference"}

FEATURES = None                # None = standard shape measurements your table has, plus PC1 to PC3
LOG_FEATURES = []              # features drawn on a log scale in the violins (does not change the statistics)
N_PCS_FOR_SEPARATION = 10      # how many principal components to use for the separation in section 8

LABELS_TIFF = None             # optional: labels volume of ONE stack, to draw typical cells
LABELS_STACK_NAME = None
VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)
MESHES_DIR = None              # optional: <MESHES_DIR>/<stack_name>/cell_<label>.obj
N_REPRESENTATIVES = 5          # typical cells per group

# ---------------- Settings with defaults ----------------
N_BOOT = 1000                  # bootstrap repeats per feature (more = smoother intervals, slower)
MAX_CELLS_FOR_BOOTSTRAP = 20000   # a group larger than this is cut to a random subset for the bootstrap only (None = never)
REPLICATE_FEATURE = None       # feature shown in the per-stack plot of section 11 (None = the first feature)
RANDOM_SEED = 42
MIN_GROUP_CELLS = 30           # a group with fewer cells is refused
SAVE_FIGURES = True

## 2. Load the table

In [ ]:
out_dir = Path(OUTPUT_DIR)
fig_dir = out_dir / "figures"
tab_dir = out_dir / "comparison_tables"
tab_dir.mkdir(parents=True, exist_ok=True)

df = h.read_table_csv(EMBEDDING_CSV)
df = h.attach_metadata(df, METADATA_CSV)
for need in ("stack_name", "label"):
    if need not in df.columns:
        raise ValueError(f"The table has no '{need}' column.")
stacks = sorted(df["stack_name"].unique())
pc_cols = [f"PC{i}" for i in range(1, N_PCS_FOR_SEPARATION + 1) if f"PC{i}" in df.columns]
print(f"{len(df):,} cells in {len(stacks)} stack(s); {len(pc_cols)} principal components found.")

DEFAULT_FEATURES = ["sphericity", "true_sphericity", "solidity", "aspect_ratio", "z_elongation", "xy_elongation",
                    "inertia_elongation", "inertia_flatness", "fill_ratio", "extent", "equivalent_diameter",
                    "surface_area", "axis_major_length", "axis_minor_length", "major_axis_z_angle",
                    "apical_basal_area_ratio"]
wanted = list(FEATURES) if FEATURES else DEFAULT_FEATURES + [c for c in ("PC1", "PC2", "PC3") if c in df.columns]
features = [f for f in wanted if f in df.columns and df[f].nunique() > 1]
skipped = [f for f in wanted if f not in features]
if skipped:
    print(f"Skipped (not in the table, or constant): {skipped}")
if not features:
    raise ValueError("No usable features. Set FEATURES in the EDIT cell.")
print(f"{len(features)} features compared.")


def safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))


def finish(fig, name):
    """Save the figure (SVG + PNG) and show it."""
    fig.tight_layout()
    if SAVE_FIGURES:
        h.save_fig(fig, name, fig_dir)
    plt.show()


def save_table(tab, name, index=True):
    tab.to_csv(tab_dir / f"{name}.csv", index=index)
    print(f"  table -> {tab_dir.name}/{name}.csv")

## 3. Look before you choose

If you define groups by cluster numbers, this is where you find them. The table
lists every cluster of `OVERVIEW_COLUMN`: how many cells, how many stacks it
occurs in, and the median of a few measurements. Pick the clusters that match
what you want to compare, then type their numbers in `GROUP_SPEC` and run again
from the top. Cluster numbers can change when the first notebook is run again,
so check them each time.

In [ ]:
if OVERVIEW_COLUMN in df.columns:
    overview = h.cluster_overview(df, OVERVIEW_COLUMN, ["sphericity", "aspect_ratio", "equivalent_diameter", "centroid-0"])
    save_table(overview.round(4), "cluster_overview", index=False)
else:
    overview = None
    print(f"No column '{OVERVIEW_COLUMN}' in the table. Columns you can use for GROUP_COLUMN or rules include:")
    print(list(df.columns)[:40])
overview

## 4. Build the groups and check them

Every cell gets the name of its group, or stays empty if it belongs to none.
The notebook refuses groups that overlap or that have fewer than
`MIN_GROUP_CELLS` cells, and it shows in which stacks each group sits.

In [ ]:
spec = {role: how for role, how in GROUP_SPEC.items() if how is not None and not (hasattr(how, "__len__") and len(how) == 0)}
if "A" not in spec or "B" not in spec:
    raise SystemExit("Stopped on purpose: define at least groups A and B in GROUP_SPEC (the EDIT cell). "
                     "Section 3 above lists your clusters. Then run again from the top.")

used_names = [GROUP_NAMES[r] for r in spec]
if len(set(used_names)) != len(used_names):
    raise ValueError(f"GROUP_NAMES must give every group its own name; got {used_names}.")
roles = h.build_groups(df, GROUP_MODE, spec, GROUP_COLUMN, GROUP_CSV)
df["group"] = roles.map(GROUP_NAMES)
role_order = [r for r in ("A", "A_ref", "B", "B_ref") if r in spec]
present = [GROUP_NAMES[r] for r in role_order]
counts = df["group"].value_counts().reindex(present).fillna(0).astype(int)
print(counts.to_string())
too_small = [g for g, n in counts.items() if n < MIN_GROUP_CELLS]
if too_small:
    raise ValueError(f"These groups have fewer than {MIN_GROUP_CELLS} cells: {too_small}. Check the definitions in GROUP_SPEC.")

group_colours = {"A": "#0072B2", "A_ref": "#9CC9E8", "B": "#D55E00", "B_ref": "#F2B88F"}
group_palette = {GROUP_NAMES[r]: group_colours[r] for r in role_order}
g = df[df["group"].notna()].copy()
name_a, name_b = GROUP_NAMES["A"], GROUP_NAMES["B"]
name_ra = GROUP_NAMES["A_ref"] if "A_ref" in spec else None
name_rb = GROUP_NAMES["B_ref"] if "B_ref" in spec else None

print("\nCells per stack:")
per_stack = pd.crosstab(g["stack_name"], g["group"]).reindex(columns=present)
save_table(per_stack, "cells_per_stack")
per_stack

## 5. Summary table

For every group and feature: the number of cells, the median and the
inter-quartile range (the middle half of the cells).

In [ ]:
summary = h.summary_by_group(g, "group", present, features)
save_table(summary.round(5), "summary_by_group", index=False)
summary.pivot(index="feature", columns="group", values="median")[present].round(3)

## 6. Departure from the reference

For each feature: Cliff's delta of a group against **its own reference**, with a
bootstrap 95% interval. A positive delta means the group's cells have larger
values than their reference's. If both groups have a reference, the two
profiles are drawn together, and the notebook reports how alike they are: do the
two groups depart from their surroundings in the same way?

In [ ]:
n_tables = (1 if name_ra else 0) + (1 if name_rb else 0) + 1 + (1 if name_ra and name_rb else 0)
size_a, size_b = int(counts.max()), int(counts.sort_values().iloc[-2])
est = h.bootstrap_time_estimate(size_a, size_b, len(features), n_tables, N_BOOT, MAX_CELLS_FOR_BOOTSTRAP)
print(f"The bootstrap in sections 6 and 7 will take roughly {est / 60:.1f} minutes ({n_tables} comparisons x {len(features)} features x {N_BOOT} repeats).")

pairs = []
display_dep = None
if name_ra:
    pairs.append((name_a, name_ra))
if name_rb:
    pairs.append((name_b, name_rb))

departures = {}
for grp, ref in pairs:
    departures[f"{grp} vs {ref}"] = h.delta_table(g, "group", grp, ref, features, n_boot=N_BOOT, seed=RANDOM_SEED, max_cells=MAX_CELLS_FOR_BOOTSTRAP)

if not departures:
    print("No reference group defined (A_ref, B_ref): this section is skipped.")
else:
    save_table(pd.concat(departures, axis=1).round(5), "departure_from_reference")
    keys = list(departures)
    fig = plt.figure(figsize=(h.figure_size("half")[0] * 1.8, max(3.2, 0.3 * len(features) + 1.4)))
    ax = fig.add_subplot(111)
    h.departure_plot(departures[keys[0]], departures[keys[1]] if len(keys) > 1 else None, keys[0],
                     keys[1] if len(keys) > 1 else "", col_a=group_colours["A"], col_b=group_colours["B"], ax=ax,
                     xlabel="Cliff's delta (group vs its own reference)")
    finish(fig, "departure_from_reference")
    profile_r = None
    if len(keys) > 1:
        profile_r = float(np.corrcoef(departures[keys[0]]["delta"], departures[keys[1]]["delta"])[0, 1])
        profile_r = profile_r if np.isfinite(profile_r) else None
        same = int((np.sign(departures[keys[0]]["delta"]) == np.sign(departures[keys[1]]["delta"])).sum())
        print("Correlation between the two departure profiles across "
              f"{len(features)} features: {'undefined' if profile_r is None else format(profile_r, '.2f')}; "
              f"same direction on {same} of {len(features)} features.")
    display_dep = pd.concat(departures, axis=1).round(3)
display_dep

## 7. Direct comparison: A against B

The same statistic between the two groups. If both groups have a reference, the
two references are compared as well. `excess_over_references` is the delta between the
groups minus the delta between their references (signed): a positive value means the groups
differ more, in that direction, than their references do. `same_direction` says whether the
two deltas have the same sign. `CIs_overlap` says whether the two intervals overlap; overlapping
intervals do not show that the two effects are equal, and separate intervals are a stricter sign than a
difference needs, so use it as a rough guide, not as a test.

In [ ]:
direct = {f"{name_a} vs {name_b}": h.delta_table(g, "group", name_a, name_b, features, n_boot=N_BOOT, seed=RANDOM_SEED, max_cells=MAX_CELLS_FOR_BOOTSTRAP)}
if name_ra and name_rb:
    direct[f"{name_ra} vs {name_rb}"] = h.delta_table(g, "group", name_ra, name_rb, features, n_boot=N_BOOT, seed=RANDOM_SEED, max_cells=MAX_CELLS_FOR_BOOTSTRAP)
table = pd.concat(direct, axis=1)
if len(direct) == 2:
    k1, k2 = list(direct)
    table[("compare", "excess_over_references")] = direct[k1]["delta"] - direct[k2]["delta"]
    table[("compare", "same_direction")] = np.sign(direct[k1]["delta"]) == np.sign(direct[k2]["delta"])
    table[("compare", "CIs_overlap")] = h.ci_overlap(direct[k1], direct[k2])
save_table(table.round(5), "direct_comparison")

fig = plt.figure(figsize=(h.figure_size("half")[0] * 1.8, max(3.2, 0.3 * len(features) + 1.4)))
ax = fig.add_subplot(111)
keys = list(direct)
h.departure_plot(direct[keys[0]], direct[keys[1]] if len(keys) > 1 else None, keys[0],
                 keys[1] if len(keys) > 1 else "", col_a="#222222", col_b="#999999", ax=ax,
                 xlabel="Cliff's delta (first group minus second)")
finish(fig, "direct_comparison")
table.round(3)

## 8. Separation in shape space (PC space)

One number per pair of groups: the distance between the groups' medians in PC
space, divided by how spread out the cells of the two groups are (the average of
each group's median distance to its own median). The medians are taken one PC at
a time. The value grows with the number of PCs used, so it has no fixed meaning on its own:
compare only pairs computed with the same PCs, above all the pair A and B against the pair of their references.
("PC space" means the space of the principal components, `PC1`, `PC2`..., that the first notebook computed from the shape measurements.)

In [ ]:
key = None
if len(pc_cols) < 2:
    print("Fewer than 2 PCs found or selected (see N_PCS_FOR_SEPARATION): this section is skipped.")
else:
    raw, norm, spread = h.pc_separation(g, "group", present, pc_cols)
    save_table(raw.round(5), "pc_distance_raw")
    save_table(norm.round(5), "pc_distance_normalised")
    print("Spread of each group:", {k: round(v, 2) for k, v in spread.items()})
    fig, ax = plt.subplots(figsize=h.figure_size("half"))
    h.matrix_heatmap(norm, ax=ax, vmin=0, cmap="Blues", cbar_label="distance / spread")
    ax.set_title(f"Separation in {len(pc_cols)}-D PC space", fontsize=h.FONT_TITLE)
    finish(fig, "pc_separation")
    key = pd.Series({f"{name_a} vs {name_b}": norm.loc[name_a, name_b]}, name="normalised distance")
    if name_ra and name_rb:
        key[f"{name_ra} vs {name_rb}"] = norm.loc[name_ra, name_rb]
    if name_ra:
        key[f"{name_a} vs {name_ra}"] = norm.loc[name_a, name_ra]
    if name_rb:
        key[f"{name_b} vs {name_rb}"] = norm.loc[name_b, name_rb]
    save_table(key.round(5).to_frame(), "pc_separation_key")
key

## 9. Violins

For each feature, the distribution in every group (violin, white dot = median,
black bar = middle half of the cells). Features listed in `LOG_FEATURES` are drawn
after log(1 + value); the statistics above do not change.

In [ ]:
fig = h.violin_grid(g, features, "group", palette=group_palette, order=present, ncols=4,
                    transforms={f: "log1p" for f in LOG_FEATURES}, rotate=25)
finish(fig, "violins")

## 10. Where are the groups on the map?

The groups in colour on top of all cells in grey. Left to right in the figure
below: all groups together, then one small map per group.

In [ ]:
if {"umap_1", "umap_2"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=h.figure_size("full"))
    h.umap_highlight(df, "umap_1", "umap_2", "group", present, ax=ax, palette=group_palette, on_top=[name_a, name_b],
                     title="Groups on the shape map")
    finish(fig, "map_groups")
    fig = h.highlight_grid(df, "umap_1", "umap_2", "group", present, palette=group_palette, ncols=len(present))
    finish(fig, "map_groups_grid")
else:
    print("The table has no umap_1 / umap_2 columns: the map is skipped.")

## 11. Do the groups repeat across stacks? (several stacks only)

A difference seen in one stack could be a quirk of that stack. When the table
holds several stacks, the first plot shows, for every group, the share of each
stack's cells (all cells of the stack, in a group or not) that belong to it (one dot per stack; a stack without the group shows a dot at 0). The second shows the median
of one feature per stack. A result that holds in most stacks is more convincing
than one carried by a single stack. Set `REPLICATE_FEATURE` to look at another feature.
With one stack this section is skipped.

In [ ]:
if len(stacks) < 2:
    print("One stack in the table: the replicate plots are skipped.")
else:
    rep_feature = REPLICATE_FEATURE or features[0]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(h.figure_size("full")[0] * 1.5, h.figure_size("full")[1]))
    _, t1 = h.replicate_dotplot(df, "group", ax=a1, groups=present, palette=group_palette)
    _, t2 = h.replicate_dotplot(df, "group", value_col=rep_feature, ax=a2, groups=present, palette=group_palette)
    save_table(t1, "share_per_stack", index=False)
    save_table(t2, f"median_{safe(rep_feature)}_per_stack", index=False)
    finish(fig, "replicates_per_stack")

## 12. Typical cells of each group (optional)

For each group, the cells closest to the group's median in PC space. They are
listed in `typical_cells.csv`. Pictures need `LABELS_TIFF` (one stack) or
`MESHES_DIR`; without them the picture is skipped.

In [ ]:
if len(pc_cols) < 2:
    print("Fewer than 2 PCs found or selected: typical cells cannot be found.")
else:
    rows = []
    for grp in present:
        sub = g[g["group"] == grp]
        X = sub[pc_cols].to_numpy(dtype=float)
        dist = np.linalg.norm(X - np.nanmedian(X, axis=0), axis=1)
        for rank, j in enumerate(np.argsort(dist)[:N_REPRESENTATIVES]):
            r = sub.iloc[j]
            rows.append({"cluster": grp, "rank": rank, "stack_name": r["stack_name"], "label": int(r["label"]),
                         "dist_to_median": float(dist[j])})
    reps = pd.DataFrame(rows)
    save_table(reps, "typical_cells", index=False)
    if LABELS_TIFF is None and MESHES_DIR is None:
        print("No LABELS_TIFF or MESHES_DIR given: pictures skipped.")
    else:
        labels_volume = h.read_labels_volume(LABELS_TIFF) if LABELS_TIFF else None
        labels_stack = LABELS_STACK_NAME or (stacks[0] if len(stacks) == 1 else None)
        fig = h.plot_cluster_gallery(reps, group_palette, labels_volume, labels_stack, VOXEL_SIZE_ZYX, MESHES_DIR,
                                     n_per_cluster=N_REPRESENTATIVES, title_prefix="")
        if fig is None:
            print("No mesh could be made for the typical cells (check the stack name and the folder).")
        else:
            finish(fig, "typical_cells_gallery")

## 13. Summary in words

The sentences below are filled in from the numbers above. A feature is listed when its
interval excludes 0 and |delta| is at least 0.147 (small). They describe the
sizes of the differences and nothing else. What a difference *means* is for you
to judge, with the caveats at the top of this notebook.

In [ ]:
def departs(dep):
    """Features where the interval excludes 0 and the effect is at least small."""
    hit = dep[((dep["ci_low"] > 0) | (dep["ci_high"] < 0)) & (dep["delta"].abs() >= 0.147)]
    return [f"{f} ({'higher' if r['delta'] > 0 else 'lower'}, {r['size']}, delta = {r['delta']:.2f})"
            for f, r in hit.sort_values("delta", key=abs, ascending=False).iterrows()]


print("SUMMARY (descriptive; effect sizes with bootstrap 95% intervals; no cell-level p-values)\n")
print("Cells: " + ", ".join(f"{k} {v:,}" for k, v in counts.items()) + f"; {len(stacks)} stack(s).")
for key_, dep in departures.items():
    print(f"\n{key_}: the groups differ on")
    for s in departs(dep) or ["no feature beyond a negligible shift"]:
        print("   -", s)
if len(departures) == 2 and profile_r is not None:
    word = ("largely AGREE" if profile_r > 0.5 else "point in OPPOSITE directions" if profile_r < 0 else "agree only partly")
    print(f"\nThe two departure profiles {word} (correlation {profile_r:.2f}).")
k_direct = list(direct)[0]
print(f"\n{k_direct}: the groups differ on")
for s in departs(direct[k_direct]) or ["no feature beyond a negligible shift"]:
    print("   -", s)
if len(pc_cols) >= 2:
    print(f"\nIn {len(pc_cols)}-D shape space the normalised distance between {name_a} and {name_b} is {norm.loc[name_a, name_b]:.2f}"
          + (f", between their references {norm.loc[name_ra, name_rb]:.2f}." if name_ra and name_rb else "."))
if len(stacks) > 1:
    print("\nLook at section 11: does the difference appear in most stacks, or in only one?")
else:
    print("\nOne stack: whether this difference repeats in other samples cannot be judged here.")
if GROUP_MODE == "column" and re.search(r"leiden|cluster|niche", GROUP_COLUMN or ""):
    print("\nReminder: the groups were picked from clusters built from shape measurements, so some separation is "
          "expected by construction." + (" Read the comparisons relative to the references." if departures else
          " Without a reference group there is nothing to compare it against; add A_ref and B_ref if you can."))
print("\nReminder: the intervals reflect the sampling of cells, not variation between samples.")

## 14. What was written

In [ ]:
for sub in (tab_dir, fig_dir):
    names = sorted(p.name for p in sub.glob("*")) if sub.exists() else []
    print(f"{sub.name}/ ({len(names)} files):")
    for n in names:
        print("   ", n)